[ Pytorch DL MODEL 실습 ]

- 데이터셋 : iris.csv
- 학습방법 : 지도학습 + 분류 ==> 2진분류


[1] 모듈로딩 <hr>

In [6]:
## 모듈로딩
import pandas as pd  # 데이터 모듈
import numpy as np

import torch         
# tensor 및 기본 함수 모듈   
import torch.nn as nn
# 인공신경망 관련 모듈
import torch.nn.functional as F
# 인공신경망 관련 함수
import torch.optim as optim
# 최적화 모듈

from sklearn.model_selection import train_test_split
# 학습용 데이터셋 관련 함수

from torch.utils.data import Dataset, DataLoader             # 인공신경망 데이터 관련 클래스 및 함수들


In [7]:
irisDF = pd.DataFrame(pd.read_csv('../_data/iris.csv', engine='python'))

[2] 데이터 로딩 및 확인 <hr> 

In [8]:
irisDF.head()
irisDF.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal.length  150 non-null    float64
 1   sepal.width   150 non-null    float64
 2   petal.length  150 non-null    float64
 3   petal.width   150 non-null    float64
 4   variety       150 non-null    object 
dtypes: float64(4), object(1)
memory usage: 6.0+ KB


In [9]:
irisDF.variety.unique()

array(['Setosa', 'Versicolor', 'Virginica'], dtype=object)

In [10]:
irisDF.variety = irisDF.variety.replace({'Setosa':0, 'Versicolor':1, 'Virginica':2})

C:\Users\kdt\AppData\Local\Temp\ipykernel_25424\292886062.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  irisDF.variety = irisDF.variety.replace({'Setosa':0, 'Versicolor':1, 'Virginica':2})


In [11]:
irisDF.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal.length  150 non-null    float64
 1   sepal.width   150 non-null    float64
 2   petal.length  150 non-null    float64
 3   petal.width   150 non-null    float64
 4   variety       150 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 6.0 KB


In [12]:
featureDF = irisDF[irisDF.columns[:-1]] # feature 4개
targetSR = irisDF[irisDF.columns[4]]  # 



In [13]:

X_train, X_test, y_train, y_test = train_test_split(
    featureDF, targetSR, train_size= 0.8, stratify=targetSR, random_state=42)

In [14]:
print(f"X_train => {X_train.ndim}D {X_train.shape} / X_test => {X_test.ndim}D, {X_test.shape}")
print(f"y_train => {y_train.ndim}D {y_train.shape}, / y_test => {y_test.ndim}D, {y_test.shape}")

X_train => 2D (120, 4) / X_test => 2D, (30, 4)
y_train => 1D (120,), / y_test => 1D, (30,)


[4] 커스텀 데이터셋 클래스 설계 및 구현 <hr>

In [39]:
# iris 전용 데이터셋 클래스#
class IrisDataset(Dataset):
    #피쳐와 타겟 분리 및 전처리 진행
    def __init__(self, featureDF, targetDF):
        super().__init__()
        self.feature = featureDF
        self.target = targetDF
        self.rows = featureDF.shape[0]
        self.cols = featureDF.shape[1]
        
    # 데이터셋의 샘플 수 반환 메서드
    def __len__(self):
        return self.rows
        
    # DataLoader 에서 batch_size만큼 호출하는 메서드
    # 인덱스에 해당하는 피쳐와 타겟 반환, 단 Tensor 형태
    def __getitem__(self, index):
        print('__getitem__()', index)
        arrFeature = self.feature.iloc[index].values
        arrTarget = self.target.values[index].reshape(-1)  
                                                        # 1D -> 2D
                                                        # 이 코드에서의 y_train, y_test는 1차원 시리즈이므로.
        return torch.FloatTensor(arrFeature), torch.FloatTensor(arrTarget)

In [40]:
""" 
넣기전에 미리 프레임으로 만들어서 넣든지
아니면 텐서로 만들어 넣든지 그건 개인 재량.
"""

' \n넣기전에 미리 프레임으로 만들어서 넣든지\n아니면 텐서로 만들어 넣든지 그건 개인 재량.\n'

In [47]:
X_train.iloc[1]

sepal.length    4.9
sepal.width     2.5
petal.length    4.5
petal.width     1.7
Name: 106, dtype: float64

In [48]:
X_test.iloc[1]

sepal.length    6.1
sepal.width     3.0
petal.length    4.9
petal.width     1.8
Name: 127, dtype: float64

In [ ]:
## train dataset은 필수
## test dataset은 선택 ==> 데이터가 많다면 DS를 만들고, DL로 생성해서 불러오는게 좋다.
trainDS = IrisDataset(X_train,y_train)
testDS = IrisDataset(X_test,y_test)
trainDS[1], testDS[1]


__getitem__() 1
__getitem__() 1


((tensor([4.9000, 2.5000, 4.5000, 1.7000]), tensor([2.])),
 (tensor([6.1000, 3.0000, 4.9000, 1.8000]), tensor([2.])))

In [51]:
## DataLoader로 확인
trainDL = DataLoader(trainDS, batch_size=3)
for feature, target in trainDL:
    print( feature, target, sep='\n')
    break

__getitem__() 0
__getitem__() 1
__getitem__() 2
tensor([[4.4000, 2.9000, 1.4000, 0.2000],
        [4.9000, 2.5000, 4.5000, 1.7000],
        [6.8000, 2.8000, 4.8000, 1.4000]])
tensor([[0.],
        [2.],
        [1.]])


In [52]:
##확인 ## DataLoader로 확인
testDL = DataLoader(testDS, batch_size=3)
for feature, target in testDL:
    print( feature, target, sep='\n')
    break

__getitem__() 0
__getitem__() 1
__getitem__() 2
tensor([[4.4000, 3.0000, 1.3000, 0.2000],
        [6.1000, 3.0000, 4.9000, 1.8000],
        [4.9000, 2.4000, 3.3000, 1.0000]])
tensor([[0.],
        [2.],
        [1.]])
